In [388]:
JSON_FILENAME = 'earthquakes_big.geojson.json'

In [389]:
import pandas as pd

Let’s read our file.

In [390]:
df = pd.read_json(JSON_FILENAME, lines=True)
df.iloc[0]

type                                                    Feature
properties    {'mag': 0.8, 'place': '6km W of Cobb, Californ...
geometry      {'type': 'Point', 'coordinates': [-122.7955, 3...
id                                                   nc72001620
Name: 0, dtype: object

We notice the type column is always equal to "Feature". Indeed:

In [391]:
df['type'].unique()

array(['Feature'], dtype=object)

So we can drop it.

In [392]:
df.drop('type', axis=1, inplace=True)

We notice some types are nested. We can flatten the data frame.

In [393]:
df_properties = pd.json_normalize(df['properties'])

We do the same for `geometry`.

In [394]:
df_geometry = pd.json_normalize(df['geometry'])


We notice the `type` column always hold the same value.

In [395]:
df_geometry['type'].unique()

array(['Point'], dtype=object)

So we can drop the `type` column.

In [396]:
df_geometry.drop('type', axis=1, inplace=True)

We can merge all this in a flattened dataframe.

In [397]:
merged_df = pd.concat([df.drop(['properties', 'geometry'], axis=1), df_properties, df_geometry], axis=1)

By listing all unique values of each column, we can infer the desired type.

In [399]:
merged_df['alert'].unique()

array([None, 'green', 'yellow'], dtype=object)

In [400]:
merged_df['code'].unique()

array(['72001620', '72001615', '10729211', ..., '71985246', '10709349',
       '2013pucw'], shape=(7669,), dtype=object)

And so on and so on. After checking the type of values for all columns, we can apply the desired type.

In [401]:
merged_df[['alert', 'code', 'detail', 'id', 'magType', 'place', 'net', 'url', 'status', 'type']] = merged_df[['alert', 'code', 'detail', 'id', 'magType', 'place', 'net', 'url', 'status', 'type']].astype(pd.StringDtype())
merged_df['time'] = merged_df['time'].astype(pd.Int64Dtype())
merged_df['types'] = merged_df['types'].apply(lambda x: x[1:-1].split(','))
merged_df['sources'] = merged_df['sources'].apply(lambda x: x[1:-1].split(','))
merged_df['ids'] = merged_df['ids'].apply(lambda x: x[1:-1].split(','))
merged_df['updated'] =  (merged_df['updated']/1000).astype('int32')

In [402]:
merged_df.rename(str.lower, axis='columns', inplace=True)
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7669 entries, 0 to 7668
Data columns (total 27 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           7669 non-null   string 
 1   mag          7668 non-null   float64
 2   place        7669 non-null   string 
 3   time         7668 non-null   Int64  
 4   updated      7669 non-null   int32  
 5   tz           7669 non-null   int64  
 6   url          7669 non-null   string 
 7   detail       7669 non-null   string 
 8   felt         801 non-null    float64
 9   cdi          801 non-null    float64
 10  mmi          91 non-null     float64
 11  alert        63 non-null     string 
 12  status       7668 non-null   string 
 13  tsunami      15 non-null     float64
 14  sig          7669 non-null   int64  
 15  net          7669 non-null   string 
 16  code         7669 non-null   string 
 17  ids          7669 non-null   object 
 18  sources      7669 non-null   object 
 19  types 

We now save our new JSON to the disk.

In [403]:
merged_df.to_json("merged_df.json", orient='records', lines=True)

## Creating the Cassandra database

We first connect to our Cassandra local cluster by running `cqlsh`.

We then create and connect to the keyspace.

    CREATE KEYSPACE ks WITH REPLICATION = { 'class': 'SimpleStrategy', 'replication_factor': 1 };
    USE ks;

We then create a table based on our `merged_df` DataFrame.

From the information above we can derive our `CREATE TABLE` statement!

    create table earthquakes (
        id text PRIMARY KEY,
        mag double,
        place text,
        time double,
        updated int,
        tz smallint,
        url text,
        detail text,
        felt double,
        cdi double,
        mmi double,
        alert text,
        status text,
        tsunami double,
        sig int,
        net text,
        code text,
        ids list<text>,
        sources list<text>,
        types list<text>,
        nst double,
        dmin double,
        rms double,
        gap double,
        magtype text,
        type text,
        coordinates tuple<double, double, double>
    );

We’ll then load our data into our keyspace. To do that, we will install and use DSBulk in our container.

 - We first download it: `wget https://github.com/datastax/dsbulk/releases/download/1.11.0/dsbulk-1.11.0.tar.gz`.
 - We extract it: `tar -zxvf dsbulk-1.11.0.tar.gz`.
 - We check we have Java installed because DSBulk requires it: `echo $JAVA_HOME`.
 - We move the folder to the `/opt` folder. `mv dsbulk-1.11.0/ /opt/`.
 - We add its path to the `PATH` environment variable. `echo 'PATH=/opt/dsbulk-1.11.0/bin/:$PATH' >> .profile`.
 - We make the shell read the new profile. `source .profile`.

And we can load our dataset into Cassandra!

    dsbulk load -c json -url merged_df.json -k ks -t earthquakes